# Power telemetry — NVML

In [1]:
import pynvml
import time
import pandas as pd

pynvml.nvmlInit()
N = pynvml.nvmlDeviceGetCount()
handles = [pynvml.nvmlDeviceGetHandleByIndex(i) for i in range(N)]
print(f"{N} GPUs found")

8 GPUs found


## Current power draw
Instantaneous reading in milliwatts — divide by 1000 for watts.

In [12]:
def get_power_draw(handles):
    return {i: pynvml.nvmlDeviceGetPowerUsage(h) / 1000 for i, h in enumerate(handles)}

power = get_power_draw(handles)
for gpu, w in power.items():
    print(f"GPU {gpu}: {w:.1f} W")

GPU 0: 66.1 W
GPU 1: 66.5 W
GPU 2: 67.3 W
GPU 3: 67.1 W
GPU 4: 68.2 W
GPU 5: 69.1 W
GPU 6: 65.9 W
GPU 7: 68.3 W


## Power limits
Four distinct values per GPU:
- **current** — what's enforced right now
- **default** — factory default
- **min / max** — range you're allowed to set via `nvmlDeviceSetPowerManagementLimit`

In [13]:
def get_power_limits(handles):
    rows = []
    for i, h in enumerate(handles):
        current  = pynvml.nvmlDeviceGetPowerManagementLimit(h) / 1000
        default  = pynvml.nvmlDeviceGetPowerManagementDefaultLimit(h) / 1000
        lo, hi   = pynvml.nvmlDeviceGetPowerManagementLimitConstraints(h)
        rows.append({"gpu": i, "current_W": current, "default_W": default,
                     "min_W": lo / 1000, "max_W": hi / 1000})
    return pd.DataFrame(rows).set_index("gpu")

get_power_limits(handles)

,current_W,default_W,min_W,max_W
gpu,,,,
0,400.0,400.0,100.0,400.0
1,400.0,400.0,100.0,400.0
2,400.0,400.0,100.0,400.0
3,400.0,400.0,100.0,400.0
4,400.0,400.0,100.0,400.0
5,400.0,400.0,100.0,400.0
6,400.0,400.0,100.0,400.0
7,400.0,400.0,100.0,400.0


## Clock throttle reasons
Bitmask that tells you *why* the GPU is running below max clock. Most interesting bits for telemetry: `SwPowerCap`, `HwSlowdown`, `HwThermalSlowdown`, `HwPowerBrakeSlowdown`.

In [15]:
THROTTLE_BITS = {
    "GpuIdle":              0x0000000000000001,
    "AppClockSetting":      0x0000000000000002,
    "SwPowerCap":           0x0000000000000004,
    "HwSlowdown":           0x0000000000000008,
    "SyncBoost":            0x0000000000000010,
    "SwThermalSlowdown":    0x0000000000000020,
    "HwThermalSlowdown":    0x0000000000000040,
    "HwPowerBrakeSlowdown": 0x0000000000000080,
    "DisplayClockSetting":  0x0000000000000100,
}

def get_throttle_reasons(handles):
    out = {}
    for i, h in enumerate(handles):
        mask = pynvml.nvmlDeviceGetCurrentClocksThrottleReasons(h)
        active = [name for name, bit in THROTTLE_BITS.items() if mask & bit]
        out[i] = active if active else ["none"]
    return out

throttle = get_throttle_reasons(handles)
for gpu, reasons in throttle.items():
    print(f"GPU {gpu}: {', '.join(reasons)}")

GPU 0: GpuIdle
GPU 1: GpuIdle
GPU 2: GpuIdle
GPU 3: GpuIdle
GPU 4: GpuIdle
GPU 5: GpuIdle
GPU 6: GpuIdle
GPU 7: GpuIdle


## Performance state (P-state)
P0 = max performance, P12 = min. Training should be pinned at P0.

In [17]:
def get_pstate(handles):
    return {i: f"P{pynvml.nvmlDeviceGetPerformanceState(h)}" for i, h in enumerate(handles)}

get_pstate(handles)

{0: 'P0', 1: 'P0', 2: 'P0', 3: 'P0', 4: 'P0', 5: 'P0', 6: 'P0', 7: 'P0'}

## Live power monitor
Polls all GPUs for `duration` seconds, prints a compact table each tick.

In [18]:
import IPython.display as ipd

def live_power(handles, duration=10, interval=1):
    header = "t  | " + "  ".join(f"GPU{i:>2}" for i in range(len(handles)))
    records = []
    for t in range(duration):
        watts = get_power_draw(handles)
        records.append({"t": t, **{f"GPU{i}": w for i, w in watts.items()}})
        row = f"{t:>2} | " + "  ".join(f"{watts[i]:>5.1f}" for i in range(len(handles)))
        if t == 0:
            print(header)
            print("-" * len(header))
        print(row)
        if t < duration - 1:
            time.sleep(interval)
    return pd.DataFrame(records).set_index("t")

df = live_power(handles, duration=10)

t  | GPU 0  GPU 1  GPU 2  GPU 3  GPU 4  GPU 5  GPU 6  GPU 7
-----------------------------------------------------------
 0 |  66.1   66.4   67.3   67.1   68.2   69.1   65.9   68.6
 1 |  66.1   66.4   67.3   66.7   68.2   69.1   65.9   68.6
 2 |  66.1   66.5   67.3   67.1   68.2   69.1   65.9   68.6
 3 |  66.1   66.4   67.3   66.7   68.2   69.1   65.9   68.6
 4 |  66.1   66.4   67.3   66.7   68.2   69.1   65.9   68.6
 5 |  66.1   66.4   67.3   66.7   68.2   69.1   65.9   68.6
 6 |  66.1   66.4   67.3   67.1   68.2   69.1   65.9   68.6
 7 |  66.1   66.4   67.3   67.0   68.2   69.1   65.9   68.6
 8 |  66.1   66.4   67.3   67.1   68.2   69.1   65.9   68.6
 9 |  66.1   66.5   67.3   67.1   68.2   69.1   65.9   68.6
